# Workspace Registry Migration

Bulk-migrates MLflow **workspace registry** models and experiments from a source workspace into the current (target) workspace under `/Shared`.

### Quick-start
1. Configure **credentials** — either a PAT token or a Service Principal (client_id / client_secret stored in Databricks Secrets).
2. Review **migration options** — prefixes, batch size, artifact handling, and optional allowlists.
3. Run **discovery** to preview what will be migrated.
4. Execute **bulk migration** to copy experiments, runs, and registered model versions to the target workspace registry.

## Prerequisites

### Network Connectivity
* The **target workspace** (where this notebook runs) must have network connectivity to the **source workspace** REST API.
* If either workspace uses a Private Link / VNet-injected deployment, ensure:
  * The target cluster/serverless compute can reach the source workspace's control-plane endpoint (port 443).
  * NSG / firewall rules allow HTTPS egress from the target to the source workspace URL.
  * If a private DNS zone is in use, the source workspace FQDN resolves correctly from the target network.
* If workspaces are in **different regions or subscriptions**, verify there is VNet peering, transit gateway, or public egress available.

### Authentication (choose one)
| Method | What to provide |
| --- | --- |
| **PAT** | Source workspace host + personal access token (stored in a Databricks secret scope recommended). |
| **Service Principal (OAuth M2M)** | Source workspace host + `client_id` + `client_secret` stored in a Databricks secret scope. The SP must be registered in the source workspace with workspace-level access. |

### Required Permissions on the Source Workspace
* Read access to workspace MLflow Tracking (experiments and runs).
* Read access to workspace Model Registry (registered models and versions).
* Artifact download permissions (if `include_run_artifacts=True`).

### Required Permissions on the Target Workspace (current)
* Write to `/Shared` experiments directory.
* Create registered models and model versions in the workspace registry.

### Secret Scope Setup (recommended)
```
# One-time setup — store credentials in Databricks Secrets
databricks secrets create-scope migration-scope
databricks secrets put-secret migration-scope source-host
databricks secrets put-secret migration-scope source-token          # if using PAT
databricks secrets put-secret migration-scope sp-client-id          # if using SP
databricks secrets put-secret migration-scope sp-client-secret      # if using SP
```

In [0]:
import importlib
import os
import sys
from dataclasses import asdict

# aDerive project root from notebook path (works for any user)
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = "/Workspace" + str(os.path.dirname(_notebook_path))
if project_root not in sys.path:
    sys.path.append(project_root)

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name.startswith("workspace_registry_migrator"):
        sys.modules.pop(module_name, None)

import workspace_registry_migrator.framework as framework
framework = importlib.reload(framework)

MigrationOptions = framework.MigrationOptions
SourceWorkspaceCredentials = framework.SourceWorkspaceCredentials
build_migrator = framework.build_migrator


In [0]:
from databricks.sdk import WorkspaceClient

# --------------------------------------------------------------------------
# AUTH MODE: Choose "pat" or "service_principal"
# --------------------------------------------------------------------------
AUTH_MODE = "service_principal"  # <-- Change to "service_principal" or pat to use SP credentials

# Source workspace credentials
# Defaults to current workspace context; replace with remote workspace values for real migrations.
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
SOURCE_HOST = ""  # _ctx.apiUrl().get()  # Replace with remote workspace URL for real migrations
SOURCE_TOKEN =""  # _ctx.apiToken().get()  # Replace with remote workspace PAT for real migrations
SP_CLIENT_ID = ""  # Service Principal client ID (if using SP)
SP_CLIENT_SECRET = ""  # Service Principal client secret (if using SP)

# --------------------------------------------------------------------------
current_workspace = WorkspaceClient()

if AUTH_MODE == "service_principal":
    credentials = SourceWorkspaceCredentials(
        host=SOURCE_HOST,
        token=None,
        client_id=SP_CLIENT_ID,
        client_secret=SP_CLIENT_SECRET,
        tracking_uri="databricks",
        registry_uri="databricks",
    )
else:
    credentials = SourceWorkspaceCredentials(
        host=SOURCE_HOST,
        token=SOURCE_TOKEN,
        tracking_uri="databricks",
        registry_uri="databricks",
    )

target_host = current_workspace.config.host

if credentials.host.rstrip("/") == target_host.rstrip("/"):
    print("\u26a0\ufe0f  WARNING: Source and target are the SAME workspace. "
          "Replace SOURCE_HOST/SOURCE_TOKEN with the remote workspace for a real migration.")

try:
    _auth = credentials.auth_type()
except ValueError as e:
    _auth = f"⚠️ NOT CONFIGURED — {e}"

{
    "auth_mode": AUTH_MODE,
    "source_host (migrate FROM)": credentials.host,
    "target_host (migrate TO)": target_host,
    "auth_type": _auth,
}


In [0]:
import os
import warnings

os.environ.update(
    {
        "TQDM_DISABLE": "1",
        "MLFLOW_ENABLE_TQDM": "false",
        "MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR": "false",
    }
)
warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# TRACKING TABLE: Delta table to persist inventory + migration progress
# --------------------------------------------------------------------------
TRACKING_CATALOG = "sunny_uc"  # <-- Update: your Unity Catalog name
TRACKING_SCHEMA = "demo"       # <-- Update: target schema
TRACKING_TABLE_NAME = "mlflow_migration_tracking"  # <-- Update if needed
TRACKING_TABLE = f"{TRACKING_CATALOG}.{TRACKING_SCHEMA}.{TRACKING_TABLE_NAME}"

# --------------------------------------------------------------------------
# MIGRATION OPTIONS
# --------------------------------------------------------------------------
options = MigrationOptions(
    shared_experiment_root="/Shared/mlflow-workspace-migration",
    model_name_prefix="",
    experiment_name_prefix="",
    batch_size=20,
    max_workers=20,
    download_artifacts=True,
    migrate_experiments=True,
    migrate_registered_models=True,
    include_run_artifacts=True,
    include_deleted_runs=True,
    max_runs_per_experiment=None,
    max_model_versions_per_model=None,
    create_missing_experiments=True,
    skip_existing_model_versions=True,
    extra_model_names=[],
    extra_experiment_ids=[],
)

print(f"Tracking table: {TRACKING_TABLE}")
asdict(options)


In [0]:
import warnings
warnings.filterwarnings("ignore")

from workspace_registry_migrator.reporting import (
    generate_inventory_report,
    print_inventory_summary,
    write_inventory_to_delta,
)

# 1. Discover source assets
migrator = build_migrator(
    source_host=credentials.host,
    source_token=credentials.token,
    source_client_id=credentials.client_id,
    source_client_secret=credentials.client_secret,
    tracking_uri=credentials.tracking_uri,
    registry_uri=credentials.registry_uri,
    tracking_table=TRACKING_TABLE,
    **asdict(options),
)
inventory = migrator.discover()

# 2. Generate inventory report
# include_metadata=False skips artifact downloads (flavors/requirements) — ~10x faster for large registries
inventory_df = generate_inventory_report(
    inventory,
    source_host=credentials.host,
    target_host=target_host,
    source_context=migrator.source,
    include_metadata=False,   # Set True to extract flavors + requirements (slower: downloads per-version artifacts)
    max_workers=20,           # Parallel threads for model processing
)
print_inventory_summary(inventory_df)

# 3. Persist to Delta tracking table (MERGE — preserves target-side columns)
write_inventory_to_delta(inventory_df, TRACKING_TABLE)

# 4. Display tracking table
display(spark.table(TRACKING_TABLE))


In [0]:
migrator = build_migrator(
    source_host=credentials.host,
    source_token=credentials.token,
    source_client_id=credentials.client_id,
    source_client_secret=credentials.client_secret,
    tracking_uri=credentials.tracking_uri,
    registry_uri=credentials.registry_uri,
    tracking_table=TRACKING_TABLE,
    **asdict(options),
)
migrator.target_workspace.workspace.mkdirs(options.shared_experiment_root)

# migrate_pending() reads from the tracking table — only processes PENDING models.
# Use migrate_all() instead if you want to re-discover and migrate everything from scratch.
asdict(migrator.migrate_pending())


In [0]:
# Live migration tracking — re-run this cell anytime to see current state
tracking_df = spark.table(TRACKING_TABLE).orderBy("migration_status", "model_name")

# Summary
from pyspark.sql.functions import col, count, sum as _sum
status_counts = tracking_df.groupBy("migration_status").agg(count("*").alias("models")).collect()
print("MIGRATION TRACKING STATUS")
for row in status_counts:
    print(f"  {row['migration_status']}: {row['models']} models")
print()

display(tracking_df)